# LEGAL MCQ - Difficulty Assignment

Assigns `difficulty` (Easy / Medium / Hard) to a balanced draw from the four
BhashaBench files in `Original Datasets/`.

| file | rows | quota |
|---|---|---|
| BhashaBench_Legal_en.jsonl | 2000 | 500 |
| BhashaBench_Legal_hi.jsonl | 2000 | 500 |
| BhashaBench_Finance_en.jsonl | 2000 | 500 |
| BhashaBench_Finance_hi.jsonl | 2000 | 500 |

**One notebook, four files.** All four are the same task - a four-option
question with a letter gold - so they share one prompt, one metric and one
pass. Sampling a fixed quota from each keeps Legal/Finance and Hindi/English
balanced at 50/50 in the output, which a single pooled draw would not.

**Protocol.** Three models answer each question. The answer is read as an
argmax over the option-letter token ids in one forward pass, so an off-list
answer is structurally impossible and no parsing is involved. Votes sum:

| Correct | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

**Two things this notebook fixes on the way through.** The Finance files store
`region` as the *string* `"null"` rather than JSON `null`; and `task_type`
carries seven question-format values (`Fill in the blanks`, `Match the
column`, ...) where the task is uniformly multiple choice. Both are normalised
in the output. The original `task_type` is kept in the audit file.

**Rows the notebook drops**, with counts printed: any row whose `answer` names
no option, and any row where one option string is repeated - if the repeat is
the gold the item has no unique answer, if it is a distractor the item is
degraded. There are 57 such rows across the four files.

**Output.** `MCQ.jsonl` - 14 schema fields with `difficulty` filled in - plus
`MCQ_audit.jsonl` with every model's pick, the gold, and BhashaBench's own
difficulty label for comparison.

**Runtime.** 2000 rows x 3 models = 6000 single forward passes, roughly 40-60
minutes on a free T4. Progress is written every batch, so a disconnect resumes
where it stopped.

### Cell 1 - Install dependencies and authenticate

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; only Mistral is
open. Accept each licence on huggingface.co, create a **read** token, then add
it in Colab via the **key icon** as a secret named `HF_TOKEN`. Use the secret
rather than pasting the token into a cell.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("HF login skipped ({}). Gated models will fail to load.".format(e))

### Cell 2 - Mount Drive

Weights are cached to Drive as 4-bit copies. The first run downloads and
quantises; every run after that loads straight from Drive, which saves about
20 GB of download and several minutes per model.

In [ ]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

### Cell 3 - Configuration and the dataset registry

- `DATASETS` - the four input files and how many rows to draw from each. Set a
  quota to `0` to skip a file; set `QUOTA_OVERRIDE` to sample the same number
  from every file without editing the registry.
- `MODELS` - the three answerers. All three are instruct checkpoints, so they
  have chat templates; Cell 6 falls back to a completion prompt if a base
  checkpoint is ever swapped in.
- `SET_TASK_TYPE` / `SET_EVAL_METRIC` - written into every output row,
  replacing the seven inherited question-format values.

In [ ]:
import gc
import os
import json
import random
import shutil
from collections import Counter, defaultdict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- where the inputs live (change if the notebook is not beside them) ----
DATA_DIR = "."

DATASETS = {
    "legal_en": {
        "file":  "BhashaBench_Legal_en.jsonl",
        "quota": 500,
        "task":  ("Answer the multiple-choice question about Indian law, "
                  "courts and legal procedure."),
    },
    "legal_hi": {
        "file":  "BhashaBench_Legal_hi.jsonl",
        "quota": 500,
        "task":  ("Answer the multiple-choice question about Indian law, "
                  "courts and legal procedure. It is written in Hindi."),
    },
    "finance_en": {
        "file":  "BhashaBench_Finance_en.jsonl",
        "quota": 500,
        "task":  ("Answer the multiple-choice question about finance, banking, "
                  "accounting or quantitative aptitude."),
    },
    "finance_hi": {
        "file":  "BhashaBench_Finance_hi.jsonl",
        "quota": 500,
        "task":  ("Answer the multiple-choice question about finance, banking, "
                  "accounting or quantitative aptitude. It is written in Hindi."),
    },
}

QUOTA_OVERRIDE = None      # e.g. 50 for a quick smoke test across all four

# ---- paths ----
OUTPUT_FILE = "MCQ.jsonl"
AUDIT_FILE  = "MCQ_audit.jsonl"
PROG_DIR    = "mcq_progress"

# ---- sampling ----
SEED       = 42
BATCH_SIZE = 25            # rows per save point

# ---- schema values written into every output row ----
SET_TASK_TYPE   = "MCQ"        # replaces 'Fill in the blanks', 'Match the column', ...
SET_EVAL_METRIC = "accuracy"

# ---- models ----
MODELS = [
    {"name": "mistral", "repo": "mistralai/Mistral-7B-Instruct-v0.3"},   # ungated
    {"name": "llama",   "repo": "meta-llama/Llama-3.1-8B-Instruct"},     # GATED
    {"name": "gemma",   "repo": "google/gemma-2-9b-it", "attn": "eager"},# GATED
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(PROG_DIR, exist_ok=True)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("output:", OUTPUT_FILE)
for k, v in DATASETS.items():
    q = QUOTA_OVERRIDE if QUOTA_OVERRIDE is not None else v["quota"]
    print("  {:<11} {:<32} quota {}".format(k, v["file"], q))

### Cell 4 - Load, normalise, filter, sample

Four things happen here, each reported so nothing is silently dropped.

1. **Gold resolution.** `answer` may be a letter or the option text; both are
   accepted. A row whose answer matches no option is dropped.
2. **Repeated options.** A row where one option string appears twice is
   dropped. If the repeat *is* the gold the item is unanswerable - a model
   picking the identical other option is marked wrong for the same answer.
3. **`region` normalisation.** The Finance files store the string `"null"`;
   it becomes JSON `null`.
4. **Per-file quota sampling** under one seed, so Legal/Finance and Hindi/
   English stay balanced.

BhashaBench ships its own `difficulty` column. It is kept as
`_orig_difficulty` for the comparison in Cell 10 and then overwritten - the
two scales measure different things (exam difficulty for humans vs. model
accuracy) and only the second belongs in this benchmark.

In [ ]:
LETTERS_ALL = [chr(ord("A") + i) for i in range(26)]


def gold_index(row):
    """Position of the correct option, whatever form the answer takes."""
    opts = row.get("options")
    if not isinstance(opts, list) or len(opts) < 2:
        return None
    ans = str(row.get("answer") or "").strip()
    if not ans:
        return None
    if len(ans) == 1 and ans.upper() in LETTERS_ALL[:len(opts)]:
        return LETTERS_ALL.index(ans.upper())
    stripped = [str(o).strip() for o in opts]
    if ans in stripped:
        return stripped.index(ans)
    low = [s.lower() for s in stripped]
    if ans.lower() in low:
        return low.index(ans.lower())
    return None


def normalise(row):
    """The Finance files carry the string "null" where JSON null was meant."""
    for key in ("region", "cultural_attr", "explanation"):
        v = row.get(key)
        if isinstance(v, str) and v.strip().lower() in ("null", "nan", "none", ""):
            row[key] = None
    return row


sample, dropped = [], defaultdict(Counter)
random.seed(SEED)

for tag, cfg in DATASETS.items():
    quota = QUOTA_OVERRIDE if QUOTA_OVERRIDE is not None else cfg["quota"]
    if not quota:
        continue
    path = os.path.join(DATA_DIR, cfg["file"])
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    pool = []
    for r in rows:
        r = normalise(r)
        gi = gold_index(r)
        if gi is None:
            dropped[tag]["no gold in options"] += 1
            continue
        opts = [str(o) for o in r["options"]]
        if len(set(opts)) != len(opts):
            key = ("repeated option (gold duplicated)"
                   if opts.count(opts[gi]) > 1 else "repeated option (distractor)")
            dropped[tag][key] += 1
            continue
        r["_gold"] = gi
        r["_dataset"] = tag
        r["_orig_difficulty"] = r.get("difficulty")
        pool.append(r)

    assert len(pool) >= quota, "{}: only {} usable rows, quota {}".format(
        tag, len(pool), quota)
    picked = random.sample(pool, quota)
    sample.extend(picked)
    print("{:<11} {:>5} rows -> {:>5} usable -> {:>4} sampled".format(
        tag, len(rows), len(pool), len(picked)))
    for reason, n in dropped[tag].most_common():
        print("              dropped {:>3}  {}".format(n, reason))

random.shuffle(sample)
print("\nSampled {} rows in total".format(len(sample)))
print("  by dataset : {}".format(dict(Counter(r["_dataset"] for r in sample))))
print("  by language: {}".format(dict(Counter(r["language"] for r in sample))))
print("  by source  : {}".format(dict(Counter(r["source"] for r in sample))))
print("  options per row: {}".format(dict(Counter(len(r["options"]) for r in sample))))
print("  gold position  : {}".format(
    {LETTERS_ALL[k]: v for k, v in sorted(Counter(r["_gold"] for r in sample).items())}))
print("  random guessing would score {:.1%}".format(
    sum(1.0 / len(r["options"]) for r in sample) / len(sample)))

r = sample[0]
print("\n--- example row ---")
print("  [{}] {}".format(r["_dataset"], " ".join(str(r["question"]).split())[:96]))
for i, o in enumerate(r["options"]):
    print("     {}. {}{}".format(LETTERS_ALL[i], str(o)[:70],
                                 " <- gold" if i == r["_gold"] else ""))

### Cell 5 - Build the prompt

The per-dataset `task` line is the only part that varies, so a Hindi legal
question and an English finance question get the same structure and differ
only in the sentence naming the domain. The prompt ends on `Answer with one
letter (A/B/C/D):` so the very next token is the one Cell 6 reads.

In [ ]:
def build_query(row):
    cfg  = DATASETS[row["_dataset"]]
    opts = "\n".join("{}. {}".format(LETTERS_ALL[i], str(o).strip())
                     for i, o in enumerate(row["options"]))
    letters = "/".join(LETTERS_ALL[:len(row["options"])])
    return ("{}\n\nQuestion:\n{}\n\nOptions:\n{}\n\n"
            "Answer with one letter ({}):").format(
                cfg["task"], " ".join(str(row["question"]).split()), opts, letters)


SYSTEM_PROMPT = (
    "You are answering multiple-choice questions from an Indian legal and "
    "financial benchmark.\n\n"
    "Reply with a SINGLE letter naming one of the given options. Output only "
    "that letter - no words, no punctuation, no explanation."
)


def build_completion(row):
    return SYSTEM_PROMPT + "\n\n" + build_query(row)


def build_chat_messages(row):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_query(row)}]


print("=" * 70)
print(build_completion(sample[0]))
print("=" * 70)
print("[gold: {}. {}]".format(LETTERS_ALL[sample[0]["_gold"]],
                              str(sample[0]["options"][sample[0]["_gold"]])[:60]))

### Cell 6 - Constrained answering

One forward pass per row. The logits at the final position are compared only
across the token ids of the valid option letters, and the largest wins. The
model never generates free text, so there is nothing to parse, no truncation
to handle, and no way to answer off-list - which also means a refusal or a
stray "The answer is" cannot be scored as wrong by accident.

Both `A` and ` A` are checked, because most tokenizers treat the
leading-space form as a different token. Prompts are truncated at 3072 tokens:
a handful of Reading Comprehension rows carry passages several thousand
characters long.

In [ ]:
def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def letter_token_ids(tokenizer, n):
    ids = {}
    for let in LETTERS_ALL[:n]:
        variants = set()
        for form in (let, " " + let):
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                variants.add(enc[0])
        ids[let] = sorted(variants)
    return ids


def format_prompt(tokenizer, row):
    if prompt_style(tokenizer) == "completion":
        return build_completion(row)
    msgs = build_chat_messages(row)
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        # some templates (Gemma) reject a system role - fold it into the user
        # turn rather than dropping the instructions
        merged = [{"role": "user",
                   "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]
        return tokenizer.apply_chat_template(
            merged, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def choose(model, tokenizer, tok_cache, row):
    n = len(row["options"])
    if n not in tok_cache:
        tok_cache[n] = letter_token_ids(tokenizer, n)
    ids = tok_cache[n]

    text   = format_prompt(tokenizer, row)
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=3072).to(model.device)
    logits = model(**inputs).logits[0, -1]

    best = max(LETTERS_ALL[:n],
               key=lambda let: max(logits[i].item() for i in ids[let]))
    return LETTERS_ALL.index(best)


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()

print("Answering functions defined")

### Cell 7 - Load-or-cache, and the batched runner

`load_model` checks Drive first. A cached copy is already 4-bit, so passing
`quantization_config` again would try to quantise twice - the branch below
avoids that. `trust_remote_code=False` throughout: all three are native
architectures in `transformers`, and the flag has caused loader failures
elsewhere in this project.

`run_model` writes every batch to `mcq_progress/<model>.jsonl` and reads it
back on start, so a Colab disconnect costs at most one batch. The loop is
model-outer / row-inner: three model loads for the whole run, not one per
dataset.

In [ ]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def run_model(spec, rows):
    prog = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    done = {}
    if os.path.exists(prog):
        with open(prog, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already answered".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    model, tokenizer = load_model(spec)
    tok_cache = {}
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in range(0, len(remaining), BATCH_SIZE):
        batch, results = remaining[start:start + BATCH_SIZE], []
        for row in batch:
            picked = choose(model, tokenizer, tok_cache, row)
            results.append({"id": row["id"], "picked": picked,
                            "gold": row["_gold"],
                            "correct": int(picked == row["_gold"])})
        with open(prog, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({i["id"]: i for i in results})
        acc = sum(v["correct"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | running acc {:.1%}".format(
            start // BATCH_SIZE + 1, total_batches, len(done), len(rows), acc))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

### Cell 8 - Run all three models

Safe to re-run after a disconnect: finished rows are read back from
`mcq_progress/` and a model that is already complete is skipped without being
loaded at all.

In [ ]:
answers = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    answers[spec["name"]] = run_model(spec, sample)

print("\nAll models done")

### Cell 9 - Assign difficulty and write the schema

Votes sum to a difficulty, the row is projected onto the 14 schema keys in
order, and `task_type` / `eval_metric` are set to the values from Cell 3. The
working fields (`_gold`, `_dataset`, `_orig_difficulty`) never reach the
output because the projection only copies `SCHEMA_KEYS`.

In [ ]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    if score == 2:
        return "Medium"
    return "Hard"


final_results, audit = [], []

for row in sample:
    votes = [answers[s["name"]][row["id"]]["correct"] for s in MODELS]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_TASK_TYPE:
        enriched["task_type"] = SET_TASK_TYPE
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":              row["id"],
        "dataset":         row["_dataset"],
        "difficulty":      difficulty,
        "bhashabench":     row["_orig_difficulty"],
        "orig_task_type":  row.get("task_type"),
        "votes":           votes,
        "gold":            LETTERS_ALL[row["_gold"]],
        "picks":           {s["name"]: LETTERS_ALL[answers[s["name"]][row["id"]]["picked"]]
                            for s in MODELS},
        "language":        row["language"],
        "subcategory":     row.get("subcategory"),
        "question":        " ".join(str(row["question"]).split())[:400],
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))

### Cell 10 - Verify and report

Four checks, none of which rebalance anything - they only make the result
legible.

- **Schema.** 14 keys in order, no null difficulty, no field left as the
  string `"null"`, and the question/options/answer of every row still
  byte-identical to its source file.
- **Distribution** overall and per dataset. Finance is expected to come out
  harder than Legal, and Hindi harder than English; if it does not, look at
  the per-model accuracies before trusting the labels.
- **Gold position vs difficulty.** The most common failure in this benchmark
  family: when models cannot read the question they default to one letter, and
  every item whose gold sits there is scored Easy for the wrong reason. If the
  `%Easy` column is flat across A/B/C/D the labels are about the questions. If
  one letter towers over the others, they are not.
- **Agreement with BhashaBench's own difficulty**, which measures exam
  difficulty for human candidates. Directional agreement is expected; exact
  agreement is not, and a low number here is not an error in either scale.

In [ ]:
bad_keys  = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
null_diff = [r["id"] for r in final_results if not r["difficulty"]]
str_null  = [r["id"] for r in final_results
             if any(isinstance(r[k], str) and r[k].strip().lower() == "null"
                    for k in SCHEMA_KEYS)]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {} | string-'null': {}".format(
    len(final_results), len(bad_keys), len(null_diff), len(str_null)))

by_id = {r["id"]: r for r in sample}
altered = [r["id"] for r in final_results
           if json.dumps([r["question"], r["options"], r["answer"]], ensure_ascii=False)
           != json.dumps([by_id[r["id"]]["question"], by_id[r["id"]]["options"],
                          by_id[r["id"]]["answer"]], ensure_ascii=False)]
print("question/options/answer altered: {}".format(len(altered)))

dist = Counter(r["difficulty"] for r in final_results)
print("\nDifficulty distribution:")
for lvl in ("Easy", "Medium", "Hard"):
    print("  {:<7}: {:5}  ({:.1%})".format(lvl, dist[lvl], dist[lvl] / len(final_results)))

print("\nPer model:")
for s in MODELS:
    acc = sum(v["correct"] for v in answers[s["name"]].values()) / len(sample)
    print("  {:<9} accuracy {:.1%}".format(s["name"], acc))

print("\nPer dataset:")
tab = defaultdict(Counter)
for a in audit:
    tab[a["dataset"]][a["difficulty"]] += 1
print("  {:<11} {:>5} {:>6} {:>7} {:>6}   %Hard".format(
    "dataset", "n", "Easy", "Medium", "Hard"))
for k in sorted(tab, key=lambda k: -tab[k]["Hard"] / max(sum(tab[k].values()), 1)):
    c = tab[k]; n = sum(c.values())
    print("  {:<11} {:>5} {:>6} {:>7} {:>6}   {:.0f}%".format(
        k, n, c["Easy"], c["Medium"], c["Hard"], 100 * c["Hard"] / n))

print("\nGold position vs difficulty (flat %Easy = position-independent):")
pos = defaultdict(Counter)
for a in audit:
    pos[a["gold"]][a["difficulty"]] += 1
print("  gold {:>6} {:>7} {:>8} {:>7}".format("n", "Easy", "%Easy", "%Hard"))
rates = []
for let in sorted(pos):
    c = pos[let]; n = sum(c.values())
    rates.append(100 * c["Easy"] / n)
    print("   {}   {:>6} {:>7} {:>7.1f}% {:>6.1f}%".format(
        let, n, c["Easy"], 100 * c["Easy"] / n, 100 * c["Hard"] / n))
print("  spread between the best and worst letter: {:.1f} points".format(
    max(rates) - min(rates)))

print("\nAgreement with BhashaBench's own difficulty:")
cross = defaultdict(Counter)
for a in audit:
    if a["bhashabench"]:
        cross[a["bhashabench"]][a["difficulty"]] += 1
same = sum(cross[k][k] for k in cross)
tot  = sum(sum(c.values()) for c in cross.values())
if tot:
    print("  exact match on {}/{} rows ({:.0f}%)".format(same, tot, 100 * same / tot))
    print("  {:<12} {:>6} {:>7} {:>8} {:>6}".format(
        "BhashaBench", "n", "Easy", "Medium", "Hard"))
    for k in ("Easy", "Medium", "Hard"):
        c = cross.get(k)
        if c:
            print("  {:<12} {:>6} {:>7} {:>8} {:>6}".format(
                k, sum(c.values()), c["Easy"], c["Medium"], c["Hard"]))

print("\n--- 2 sample rows ---")
for a in audit[:2]:
    print("\n  {} [{}] {} | gold {} | picks {}".format(
        a["id"][:18], a["dataset"], a["difficulty"], a["gold"], a["picks"]))
    print("    {}".format(a["question"][:120]))